In [1]:
!nvidia-smi
import torch

print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(
        f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB"
    )

Thu Jun 11 17:38:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 1: Clone Repository and Setup Environment

In [ ]:
import os
import getpass
from pathlib import Path

# Configuration
GITHUB_USER = "sattary"
REPO_NAME = "ali_proj"
BRANCH = "fix-review"  # Change if using different branch
PROJECT_DIR = "ali_proj"

print("Enter your GitHub Personal Access Token (PAT):")
PAT = getpass.getpass()
REPO_URL = f"https://{PAT}@github.com/{GITHUB_USER}/{REPO_NAME}.git"

# 1. Clone Repository
if not Path(PROJECT_DIR).exists():
    print(f"Cloning {REPO_NAME} (branch: {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}
else:
    print("Repository already cloned. Pulling latest changes...")
    !cd {PROJECT_DIR} && git pull origin {BRANCH}

%cd {PROJECT_DIR}

# 2. Install uv
print("\nInstalling uv...")
!pip install -q uv

# 3. Set MPLBACKEND for Kaggle compatibility
os.environ['MPLBACKEND'] = 'Agg'
print("\nSet MPLBACKEND=Agg for headless environments")

# 4. Sync Dependencies
print("\nSyncing dependencies...")
!uv sync

print("\n✓ Setup complete!")

Cloning https://github.com/sattary/ali_proj.git (branch: fix-review)...
Cloning into 'ali_proj'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: 'ali_proj'
/content

Installing uv...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 85.9 MB/s eta 0:00:00:00:0100:01

Set MPLBACKEND=Agg for Kaggle compatibility

Syncing dependencies...
error: No `pyproject.toml` found in current directory or any parent directory

✓ Setup complete!


In [ ]:
# Verify Git Repository
from pathlib import Path
import subprocess

# Check if we're in a git repo
result = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True
)
if result.returncode == 0:
    repo_root = result.stdout.strip()
    print(f"✓ Git repository found at: {repo_root}")
    print(f"✓ Current directory: {Path.cwd()}")
else:
    print("Error: Not in a git repository!")
    print(f"Current directory: {Path.cwd()}")
    raise RuntimeError("Git repository not found")

# Show repo status
!git status

## Phase 1: Deterministic Generation (`generate`)

Generate the synthetic interferogram dataset to HDF5 shards. The cryptographic seed guarantees mathematically invariant noise topologies.

In [ ]:
# Phase 1: Generate Data
!uv run phase-unwrap generate \
    --num-samples 180000 \
    --shard-size 1000 \
    --out-dir data/kaggle_full \
    --seed 1337

print("\n✓ Data generation complete!")

## Phase 2: Hyperparameter Optimization (`tune`)

Use Optuna's Bayesian TPE algorithm to isolate the absolute lowest-error configuration. The best configuration is automatically saved to `runs/optuna/best_config.yaml`.

In [ ]:
# Phase 2: Optuna Tune
# Automatically parallelizes across available GPUs
!uv run phase-unwrap tune \
    --use-amp \
    --n-trials 30 \
    --tune-epochs 15 \
    --study-name kaggle_hpo \
    --n-workers 1 \
    --batch-size 16

print("\n✓ Tuning complete!")
print("Best config frozen to: runs/optuna/best_config.yaml")

## Phase 3: Primary Training and Evaluation (`train`)

Train the baseline network to convergence using the frozen `best_config.yaml`.

## Phase 4: Statistical Validation (`multiseed`)

Defend against 'lucky seed' anomalies. Spawns completely independent training convergences using the same frozen configuration, and automatically aggregates the metrics into Mean ± Std.

## Phase 5: Architectural Ablation (`ablation`)

Mathematically prove the necessity of your custom topology by systematically crippling the network. Exports a rigorous LaTeX comparison table.

## Step 6: Zip and Download Results

Run this cell to zip the `runs/` and `results/` directories so you can render them on your local machine.

In [ ]:
import shutil
from IPython.display import FileLink

print("Zipping runs and results...")
shutil.make_archive('training_results', 'zip', 'runs/')
shutil.make_archive('tables_results', 'zip', 'results/')
print("✓ Done!")
display(FileLink('training_results.zip'))
display(FileLink('tables_results.zip'))